# ver19 Follow-up - Internal Adapter 5-Fold Validation

This experiment resumes the Internal Adapter branch that was left as a Fold-1 pilot in ver14.
The original project stopped full validation because of time constraints and prioritized multi-foundation integration.
This follow-up keeps the original ver14 configuration fixed and tests whether its Fold-1 performance generalizes across all five patient-level folds.

이 노트북은 기존 프로젝트 당시 수행한 실험을 소급해서 바꾸는 것이 아니라, **프로젝트 정리 이후 수행하는 후속 검증**입니다.

연구 질문은 하나입니다.

> ver14 Fold 1에서 관찰된 BiomedCLIP Internal Bottleneck Adapter의 높은 sensitivity가 동일한 설정의 patient-level 5-fold cross-validation에서도 재현되는가?

유지 조건:

- Task: `NonDemented` vs `Demented = VeryMild + Mild + Moderate`
- 환자 단위 5-fold, seed 42, inner validation 15%
- BiomedCLIP pretrained weight freeze
- 마지막 visual block 2개에 bottleneck adapter 삽입
- augmentation 없음 (`TRAIN_AUG_MODE="original"`)
- threshold는 각 fold의 inner validation에서만 선택
- 기존 ver14 notebook/checkpoint/result는 수정하지 않음

이 노트북에는 사전 계산된 성능값이 없습니다. 아래 결과 CSV/JSON은 실제 5-fold 학습이 완료된 뒤에만 생성됩니다.


## 1. 환경 점검 및 Seed 고정


In [ ]:
import os
import sys
import time
import gc
import json
import re
import math
import random
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)

SEED = 42


def seed_everything(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True
    os.environ["PYTHONHASHSEED"] = str(seed)


seed_everything(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"Python executable : {sys.executable}")
print(f"Python version    : {sys.version}")
print(f"torch version     : {torch.__version__}")
print(f"torch.version.cuda: {torch.version.cuda}")
print(f"cuda available    : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU name          : {torch.cuda.get_device_name(0)}")
else:
    print("GPU name          : CUDA GPU not available")
print(f"device            : {device}")


## 2. 고정 실험 설정

아래 값은 ver14 Fold 1 pilot과 동일하게 고정합니다. 이 후속 검증은 hyperparameter 탐색이 아니므로 fold별 변경을 허용하지 않습니다.

`DATA_ROOT`만 실행 PC의 데이터셋 위치에 맞게 확인하세요. 환경변수 `ALZHEIMER_DATA_ROOT`가 있으면 그 값을 우선 사용합니다.


In [ ]:
def find_repository_root(start=None):
    current = Path(start or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "notebooks").is_dir() and (candidate / "results").is_dir():
            return candidate
    fallback = Path(r"C:\Users\user\Desktop\alzheimer\DL_project_github")
    assert fallback.exists(), "GitHub repository root를 찾지 못했습니다."
    return fallback


REPO_ROOT = find_repository_root()
DATA_ROOT = Path(
    os.environ.get(
        "ALZHEIMER_DATA_ROOT",
        r"C:\Users\user\Desktop\alzheimer_dataset\Data",
    )
)

OUTPUT_DIR = REPO_ROOT / "outputs" / "internal_adapter_5fold_followup"
CHECKPOINT_DIR = OUTPUT_DIR / "checkpoints"
OOF_PARTS_DIR = OUTPUT_DIR / "oof_parts"
THRESHOLD_DIR = OUTPUT_DIR / "threshold_tables"
RESULTS_DIR = REPO_ROOT / "results"

for directory in [OUTPUT_DIR, CHECKPOINT_DIR, OOF_PARTS_DIR, THRESHOLD_DIR, RESULTS_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

NEGATIVE_CLASSES = ["NonDemented"]
POSITIVE_CLASSES = ["VeryMildDemented", "MildDemented", "ModerateDemented"]
TASK_NAME = "internal_adapter_5fold_followup"
EXPERIMENT_NAME = "internal_adapter_5fold_followup"

MODEL_NAME = "hf-hub:microsoft/BiomedCLIP-PubMedBERT_256-vit_base_patch16_224"

N_SPLITS = 5
INNER_VAL_RATIO = 0.15
FOLDS_TO_RUN = list(range(1, N_SPLITS + 1))
EXPECTED_PATIENTS = 347

# ver14 Internal Adapter 설정 고정
ADAPTER_LAST_VISUAL_BLOCKS = 2
ADAPTER_HIDDEN_DIM = 64
ADAPTER_DROPOUT = 0.1

# 이번 검증은 augmentation 효과 실험이 아닙니다.
TRAIN_AUG_MODE = "original"
ROTATION_DEGREES = 10
SHIFT_TRANSLATE = (0.05, 0.05)
ZOOM_SCALE = (0.9, 1.1)
DETERMINISTIC_AUGMENTATION = True

BATCH_SIZE = 16
NUM_WORKERS = 0
PIN_MEMORY = True
PERSISTENT_WORKERS = NUM_WORKERS > 0
USE_AMP = True

EPOCHS = 5
HEAD_LR = 5e-4
ADAPTER_LR = 5e-5
WEIGHT_DECAY = 1e-4
EARLY_STOPPING_PATIENCE = 2
EARLY_STOPPING_MIN_DELTA = 1e-4
BALANCE_STRATEGY = "class_weight_sqrt"

TARGET_VALIDATION_SENSITIVITY = 0.85
MIN_VALIDATION_SPECIFICITY = 0.65
THRESHOLD_GRID = np.round(np.arange(0.10, 0.7001, 0.025), 3)

RESULTS_PATH = RESULTS_DIR / "internal_adapter_5fold_results.csv"
SUMMARY_PATH = RESULTS_DIR / "internal_adapter_5fold_summary.csv"
OOF_PREDICTIONS_PATH = RESULTS_DIR / "internal_adapter_5fold_oof_predictions.csv"
OOF_METRICS_PATH = RESULTS_DIR / "internal_adapter_5fold_oof_metrics.json"
COMPARISON_PATH = RESULTS_DIR / "internal_adapter_5fold_vs_adapter_probe.csv"
FOLD_ASSIGNMENTS_PATH = OUTPUT_DIR / "internal_adapter_5fold_assignments.csv"
CONFIG_SNAPSHOT_PATH = OUTPUT_DIR / "experiment_config.json"

# 실험 조건이 우발적으로 바뀌면 학습 전에 중단합니다.
assert SEED == 42
assert N_SPLITS == 5
assert INNER_VAL_RATIO == 0.15
assert FOLDS_TO_RUN == [1, 2, 3, 4, 5]
assert ADAPTER_LAST_VISUAL_BLOCKS == 2
assert ADAPTER_HIDDEN_DIM == 64
assert ADAPTER_DROPOUT == 0.1
assert TRAIN_AUG_MODE == "original"
assert BATCH_SIZE == 16
assert EPOCHS == 5
assert HEAD_LR == 5e-4
assert ADAPTER_LR == 5e-5
assert WEIGHT_DECAY == 1e-4
assert EARLY_STOPPING_PATIENCE == 2
assert EARLY_STOPPING_MIN_DELTA == 1e-4
assert BALANCE_STRATEGY == "class_weight_sqrt"
assert TARGET_VALIDATION_SENSITIVITY == 0.85
assert MIN_VALIDATION_SPECIFICITY == 0.65
assert np.array_equal(THRESHOLD_GRID, np.round(np.arange(0.10, 0.7001, 0.025), 3))

config_snapshot = {
    "follow_up_status": "post-project validation of deferred ver14 pilot",
    "model_name": MODEL_NAME,
    "seed": SEED,
    "n_splits": N_SPLITS,
    "inner_val_ratio": INNER_VAL_RATIO,
    "adapter_last_visual_blocks": ADAPTER_LAST_VISUAL_BLOCKS,
    "adapter_hidden_dim": ADAPTER_HIDDEN_DIM,
    "adapter_dropout": ADAPTER_DROPOUT,
    "train_aug_mode": TRAIN_AUG_MODE,
    "batch_size": BATCH_SIZE,
    "epochs": EPOCHS,
    "head_lr": HEAD_LR,
    "adapter_lr": ADAPTER_LR,
    "weight_decay": WEIGHT_DECAY,
    "early_stopping_patience": EARLY_STOPPING_PATIENCE,
    "early_stopping_min_delta": EARLY_STOPPING_MIN_DELTA,
    "balance_strategy": BALANCE_STRATEGY,
    "target_validation_sensitivity": TARGET_VALIDATION_SENSITIVITY,
    "min_validation_specificity": MIN_VALIDATION_SPECIFICITY,
    "threshold_grid": THRESHOLD_GRID.tolist(),
}
CONFIG_SNAPSHOT_PATH.write_text(
    json.dumps(config_snapshot, indent=2, ensure_ascii=False),
    encoding="utf-8",
)

print(f"REPO_ROOT : {REPO_ROOT}")
print(f"DATA_ROOT : {DATA_ROOT}")
print(f"OUTPUT_DIR: {OUTPUT_DIR}")
print(f"RESULTS_DIR: {RESULTS_DIR}")
print(f"FOLDS_TO_RUN: {FOLDS_TO_RUN}")
print(f"TRAIN_AUG_MODE: {TRAIN_AUG_MODE}")
print(
    f"Internal adapter: last_blocks={ADAPTER_LAST_VISUAL_BLOCKS}, "
    f"hidden_dim={ADAPTER_HIDDEN_DIM}, dropout={ADAPTER_DROPOUT}"
)


## 3. 환자 단위 Manifest 생성

`ver9~ver13`과 동일하게 파일명에서 `OAS1_xxxx` 환자 ID를 추출합니다.
같은 환자의 slice는 반드시 같은 split에만 들어가야 하므로 Fold는 환자 단위로 나눕니다.


In [ ]:
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
PATIENT_PATTERN = re.compile(r"^(OAS1_\d+)", re.IGNORECASE)

rows = []
selected_classes = NEGATIVE_CLASSES + POSITIVE_CLASSES

for class_name in selected_classes:
    target = 0 if class_name in NEGATIVE_CLASSES else 1
    class_dir = DATA_ROOT / class_name
    assert class_dir.exists(), f"Class directory not found: {class_dir}"

    for image_path in sorted(class_dir.iterdir()):
        if not image_path.is_file() or image_path.suffix.lower() not in IMAGE_EXTENSIONS:
            continue
        patient_match = PATIENT_PATTERN.match(image_path.name)
        assert patient_match, f"Patient ID를 추출할 수 없습니다: {image_path.name}"
        rows.append({
            "image_path": str(image_path),
            "class_name": class_name,
            "patient_id": patient_match.group(1).upper(),
            "target": target,
        })

manifest = pd.DataFrame(rows)
assert not manifest.empty
assert manifest.groupby("patient_id")["target"].nunique().max() == 1
assert manifest.groupby("patient_id")["class_name"].nunique().max() == 1

patient_table = (
    manifest.groupby("patient_id", as_index=False)
    .agg(
        target=("target", "first"),
        class_name=("class_name", "first"),
        image_count=("image_path", "count"),
    )
)

manifest_path = OUTPUT_DIR / f"{TASK_NAME}_all_images_manifest.csv"
patient_table_path = OUTPUT_DIR / f"{TASK_NAME}_patient_table.csv"
manifest.to_csv(manifest_path, index=False, encoding="utf-8-sig")
patient_table.to_csv(patient_table_path, index=False, encoding="utf-8-sig")

print(f"Images: {len(manifest):,}")
print(f"Patients: {len(patient_table):,}")
assert len(patient_table) == EXPECTED_PATIENTS, (
    f"Expected {EXPECTED_PATIENTS} patients, found {len(patient_table)}. "
    "데이터셋 경로/구성이 기존 실험과 같은지 확인하세요."
)
print("\n[Binary target patient counts]")
print(patient_table["target"].value_counts().sort_index())
print("\n[Original class patient counts]")
print(patient_table["class_name"].value_counts())


## 4. BiomedCLIP 로드 및 feature dimension 확인


In [ ]:
import open_clip


def load_biomedclip_model():
    print("BiomedCLIP 로드 중...")
    model, preprocess = open_clip.create_model_from_pretrained(MODEL_NAME)
    model = model.to(device)
    model.eval()

    model_device = next(model.parameters()).device
    print(f"model parameter device: {model_device}")
    assert model_device.type == device.type, (
        f"BiomedCLIP model is not on expected device. expected={device}, current={model_device}"
    )
    return model, preprocess


def infer_image_feature_dim(model, preprocess, sample_image_path):
    model.eval()
    image = Image.open(sample_image_path).convert("RGB")
    tensor = preprocess(image).unsqueeze(0).to(device)
    with torch.inference_mode():
        with torch.amp.autocast(
            "cuda",
            enabled=(USE_AMP and torch.cuda.is_available()),
        ):
            features = model.encode_image(tensor)
    return int(features.shape[-1])


base_model_for_check, preprocess = load_biomedclip_model()
sample_image_path = manifest.iloc[0]["image_path"]
FEATURE_DIM = infer_image_feature_dim(base_model_for_check, preprocess, sample_image_path)
print(f"FEATURE_DIM: {FEATURE_DIM}")

del base_model_for_check
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()


## 5. Dataset 정의

기본값은 `TRAIN_AUG_MODE="original"`입니다.

이번 실험은 augmentation 효과가 아니라 internal adapter 효과를 확인하는 것이 목적입니다.
따라서 먼저 원본 이미지만 사용합니다.


In [ ]:
class PatientImageDataset(Dataset):
    def __init__(
        self,
        dataframe,
        preprocess,
        aug_mode="original",
        deterministic=True,
        seed=42,
    ):
        self.df = dataframe.reset_index(drop=True).copy()
        self.preprocess = preprocess
        self.deterministic = deterministic
        self.seed = seed

        if aug_mode == "original":
            self.output_types = ["original"]
        elif aug_mode == "sepaug_4n":
            self.output_types = ["original", "rotation", "shift", "zoom"]
        else:
            raise ValueError(aug_mode)

        self.rotation = transforms.RandomRotation(ROTATION_DEGREES)
        self.shift = transforms.RandomAffine(degrees=0, translate=SHIFT_TRANSLATE)
        self.zoom = transforms.RandomAffine(degrees=0, scale=ZOOM_SCALE)

    def __len__(self):
        return len(self.df) * len(self.output_types)

    def _apply_augmentation(self, image, aug_type):
        if aug_type == "original":
            return image
        if aug_type == "rotation":
            return self.rotation(image)
        if aug_type == "shift":
            return self.shift(image)
        if aug_type == "zoom":
            return self.zoom(image)
        raise ValueError(aug_type)

    def __getitem__(self, index):
        multiplier = len(self.output_types)
        base_index = index // multiplier
        aug_index = index % multiplier
        aug_type = self.output_types[aug_index]
        row = self.df.iloc[base_index]

        image = Image.open(row["image_path"]).convert("RGB")
        if self.deterministic:
            local_seed = self.seed + base_index * 1009 + aug_index * 9176
            state_py = random.getstate()
            state_np = np.random.get_state()
            state_torch = torch.random.get_rng_state()
            random.seed(local_seed)
            np.random.seed(local_seed % (2**32 - 1))
            torch.manual_seed(local_seed)
            image = self._apply_augmentation(image, aug_type)
            random.setstate(state_py)
            np.random.set_state(state_np)
            torch.random.set_rng_state(state_torch)
        else:
            image = self._apply_augmentation(image, aug_type)

        return (
            self.preprocess(image),
            torch.tensor(int(row["target"]), dtype=torch.long),
            row["patient_id"],
        )


## 6. Internal Adapter 모듈 정의

adapter는 기존 block의 출력을 작은 bottleneck network에 통과시켜 다시 더합니다.

```text
output = frozen_block(x)
output = output + adapter(output)
```

중요한 점:

- 기존 BiomedCLIP block weight는 고정합니다.
- adapter의 마지막 projection은 0으로 초기화합니다.
- 따라서 학습 시작 시점에는 원래 BiomedCLIP 출력과 거의 동일합니다.
- 학습되는 것은 adapter와 classifier head뿐입니다.


In [ ]:
class BottleneckAdapter(nn.Module):
    def __init__(self, dim, hidden_dim=64, dropout=0.1):
        super().__init__()
        self.down = nn.Linear(dim, hidden_dim)
        self.act = nn.GELU()
        self.dropout = nn.Dropout(dropout)
        self.up = nn.Linear(hidden_dim, dim)

        nn.init.kaiming_uniform_(self.down.weight, a=math.sqrt(5))
        nn.init.zeros_(self.down.bias)
        nn.init.zeros_(self.up.weight)
        nn.init.zeros_(self.up.bias)

    def forward(self, x):
        return self.up(self.dropout(self.act(self.down(x))))


class AdapterWrappedBlock(nn.Module):
    def __init__(self, block, dim, hidden_dim=64, dropout=0.1):
        super().__init__()
        self.block = block
        for parameter in self.block.parameters():
            parameter.requires_grad = False
        self.adapter = BottleneckAdapter(dim, hidden_dim, dropout)

    def forward(self, *args, **kwargs):
        output = self.block(*args, **kwargs)
        if isinstance(output, tuple):
            main = output[0]
            return (main + self.adapter(main), *output[1:])
        return output + self.adapter(output)


def infer_block_dim(block):
    for module in block.modules():
        if isinstance(module, nn.LayerNorm):
            shape = module.normalized_shape
            if isinstance(shape, int):
                return int(shape)
            if len(shape) > 0:
                return int(shape[-1])

    candidates = []
    for module in block.modules():
        if isinstance(module, nn.Linear):
            candidates.extend([module.in_features, module.out_features])
    if candidates:
        # ViT hidden dimension is usually the smaller repeated feature dimension.
        values, counts = np.unique(candidates, return_counts=True)
        return int(values[np.argmax(counts)])

    raise RuntimeError("Block hidden dimension을 추론할 수 없습니다.")


def get_visual_block_container(clip_model):
    visual = getattr(clip_model, "visual", None)
    candidates = []
    if visual is not None:
        candidates.extend([
            ("visual.trunk.blocks", lambda: visual.trunk.blocks),
            ("visual.transformer.resblocks", lambda: visual.transformer.resblocks),
            ("visual.blocks", lambda: visual.blocks),
        ])

    for name, getter in candidates:
        try:
            blocks = getter()
            if hasattr(blocks, "__len__") and len(blocks) > 0:
                return name, blocks
        except Exception:
            pass
    return None, None


def apply_internal_adapters(
    clip_model,
    n_last_blocks=2,
    hidden_dim=64,
    dropout=0.1,
):
    block_container_name, blocks = get_visual_block_container(clip_model)
    if blocks is None:
        raise RuntimeError("BiomedCLIP visual block container를 찾지 못했습니다.")

    n_last_blocks = min(n_last_blocks, len(blocks))
    wrapped = []

    for block_index in range(len(blocks) - n_last_blocks, len(blocks)):
        original_block = blocks[block_index]
        dim = infer_block_dim(original_block)
        wrapped_block = AdapterWrappedBlock(
            original_block,
            dim=dim,
            hidden_dim=hidden_dim,
            dropout=dropout,
        )
        blocks[block_index] = wrapped_block
        wrapped.append({
            "name": f"{block_container_name}.{block_index}",
            "dim": dim,
            "hidden_dim": hidden_dim,
        })

    return wrapped


class BiomedCLIPInternalAdapterClassifier(nn.Module):
    def __init__(self, clip_model, feature_dim):
        super().__init__()
        self.clip_model = clip_model
        self.classifier = nn.Linear(feature_dim, 2)

    def forward(self, images):
        features = self.clip_model.encode_image(images)
        features = F.normalize(features.float(), dim=-1)
        return self.classifier(features)


def build_internal_adapter_classifier():
    clip_model, _ = load_biomedclip_model()

    for parameter in clip_model.parameters():
        parameter.requires_grad = False

    wrapped_blocks = apply_internal_adapters(
        clip_model,
        n_last_blocks=ADAPTER_LAST_VISUAL_BLOCKS,
        hidden_dim=ADAPTER_HIDDEN_DIM,
        dropout=ADAPTER_DROPOUT,
    )

    model = BiomedCLIPInternalAdapterClassifier(clip_model, FEATURE_DIM).to(device)
    for parameter in model.classifier.parameters():
        parameter.requires_grad = True

    trainable_names = [
        name for name, parameter in model.named_parameters()
        if parameter.requires_grad
    ]
    assert trainable_names, "학습 가능한 parameter가 없습니다."
    assert all(
        name.startswith("classifier.") or ".adapter." in name
        for name in trainable_names
    ), f"adapter/classifier 외 trainable parameter 발견: {trainable_names}"
    assert any(".adapter." in name for name in trainable_names)
    assert any(name.startswith("classifier.") for name in trainable_names)

    frozen_pretrained = [
        name for name, parameter in model.clip_model.named_parameters()
        if ".adapter." not in name and not parameter.requires_grad
    ]
    unexpected_trainable_pretrained = [
        name for name, parameter in model.clip_model.named_parameters()
        if ".adapter." not in name and parameter.requires_grad
    ]
    assert frozen_pretrained, "freeze 검증 대상 pretrained parameter가 없습니다."
    assert not unexpected_trainable_pretrained, (
        "BiomedCLIP pretrained parameter가 trainable입니다: "
        f"{unexpected_trainable_pretrained[:10]}"
    )

    print(f"Internal adapter 적용 block 수: {len(wrapped_blocks)}")
    for item in wrapped_blocks:
        print(f"  - {item['name']} | dim={item['dim']} hidden={item['hidden_dim']}")

    print(f"Trainable tensors: {len(trainable_names)}")
    print("Freeze audit passed: adapter + classifier만 학습합니다.")
    return model, wrapped_blocks


def count_trainable_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


def count_total_parameters(model):
    return sum(p.numel() for p in model.parameters())


def trainable_state_dict(model):
    trainable_names = {
        name
        for name, parameter in model.named_parameters()
        if parameter.requires_grad
    }
    return {
        name: value.detach().cpu().clone()
        for name, value in model.state_dict().items()
        if name in trainable_names
    }


## 7. 평가 및 threshold 선택 함수


In [ ]:
def safe_auroc(y_true, y_prob):
    if len(np.unique(y_true)) < 2:
        return np.nan
    return roc_auc_score(y_true, y_prob)


def safe_auprc(y_true, y_prob):
    if len(np.unique(y_true)) < 2:
        return np.nan
    return average_precision_score(y_true, y_prob)


def aggregate_patient_predictions(patient_ids, labels, probabilities, threshold=0.5):
    grouped_probs = defaultdict(list)
    grouped_labels = {}
    for patient_id, label, prob in zip(patient_ids, labels, probabilities):
        grouped_probs[patient_id].append(float(prob))
        grouped_labels[patient_id] = int(label)

    ordered_ids = sorted(grouped_probs)
    patient_probs = np.array([np.mean(grouped_probs[pid]) for pid in ordered_ids])
    patient_labels = np.array([grouped_labels[pid] for pid in ordered_ids])
    patient_preds = (patient_probs >= threshold).astype(int)
    return ordered_ids, patient_labels, patient_probs, patient_preds


def calculate_metrics(y_true, y_prob, threshold=0.5):
    y_true = np.asarray(y_true).astype(int)
    y_prob = np.asarray(y_prob).astype(float)
    y_pred = (y_prob >= threshold).astype(int)
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    specificity = cm[0, 0] / max(cm[0].sum(), 1)
    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "sensitivity": recall_score(y_true, y_pred, pos_label=1, zero_division=0),
        "specificity": specificity,
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "macro_f1": f1_score(y_true, y_pred, average="macro", zero_division=0),
        "auroc": safe_auroc(y_true, y_prob),
        "auprc": safe_auprc(y_true, y_prob),
    }


def build_threshold_table(y_true, y_prob, threshold_grid):
    rows = []
    for threshold in threshold_grid:
        metrics = calculate_metrics(y_true, y_prob, threshold=float(threshold))
        rows.append({
            "threshold": float(threshold),
            "distance_from_0_5": abs(float(threshold) - 0.5),
            **metrics,
        })
    return pd.DataFrame(rows)


def choose_sensitivity_first_threshold(
    y_true,
    y_prob,
    target_sensitivity=TARGET_VALIDATION_SENSITIVITY,
    min_specificity=MIN_VALIDATION_SPECIFICITY,
):
    table = build_threshold_table(y_true, y_prob, THRESHOLD_GRID)
    eligible = table[
        (table["sensitivity"] >= target_sensitivity)
        & (table["specificity"] >= min_specificity)
    ].copy()
    target_reached = not eligible.empty

    if not target_reached:
        eligible = table[table["specificity"] >= min_specificity].copy()
        if eligible.empty:
            eligible = table.copy()
        eligible = eligible[
            eligible["sensitivity"] == eligible["sensitivity"].max()
        ].copy()

    selected = eligible.sort_values(
        ["specificity", "precision", "distance_from_0_5"],
        ascending=[False, False, True],
    ).iloc[0]
    return float(selected["threshold"]), bool(target_reached), table


def evaluate_model(model, loader):
    model.eval()
    patient_ids, labels_all, probabilities_all = [], [], []

    with torch.inference_mode():
        for images, labels, batch_patient_ids in loader:
            images = images.to(device, non_blocking=True)
            with torch.amp.autocast(
                "cuda",
                enabled=(USE_AMP and torch.cuda.is_available()),
            ):
                logits = model(images)
                probs = logits.softmax(dim=1)[:, 1]

            patient_ids.extend(list(batch_patient_ids))
            labels_all.extend(labels.numpy().tolist())
            probabilities_all.extend(probs.detach().cpu().numpy().tolist())

    return aggregate_patient_predictions(
        patient_ids,
        labels_all,
        probabilities_all,
        threshold=0.5,
    )[:3]


def make_class_weight(inner_train_patients):
    counts_series = inner_train_patients["target"].value_counts().sort_index()
    counts = np.array(
        [counts_series.get(0, 0), counts_series.get(1, 0)],
        dtype=np.float64,
    )
    if BALANCE_STRATEGY == "class_weight_sqrt":
        values = 1.0 / np.sqrt(np.maximum(counts, 1))
        values /= values.mean()
        return torch.tensor(values, dtype=torch.float32, device=device)
    return None


## 8. Internal Adapter Fold 학습 함수

ver14의 loader, optimizer, class weighting, early stopping을 유지합니다.

구조적으로 threshold 선택은 `val_loader` 평가 직후 수행되고, 이후에만 `test_loader`를 평가합니다. 따라서 outer test prediction은 threshold 선택에 사용되지 않습니다.


In [ ]:
def make_loaders_for_fold(
    fold_number,
    outer_train_patients,
    outer_test_patients,
):
    run_seed = SEED + fold_number * 100
    inner_train_patients, inner_val_patients = train_test_split(
        outer_train_patients,
        test_size=INNER_VAL_RATIO,
        random_state=run_seed,
        stratify=outer_train_patients["target"],
    )

    inner_train_ids = set(inner_train_patients["patient_id"])
    inner_val_ids = set(inner_val_patients["patient_id"])
    outer_test_ids = set(outer_test_patients["patient_id"])

    assert inner_train_ids.isdisjoint(inner_val_ids)
    assert inner_train_ids.isdisjoint(outer_test_ids)
    assert inner_val_ids.isdisjoint(outer_test_ids)
    assert inner_train_ids | inner_val_ids == set(outer_train_patients["patient_id"])

    train_df = manifest[manifest["patient_id"].isin(inner_train_ids)].copy()
    val_df = manifest[manifest["patient_id"].isin(inner_val_ids)].copy()
    test_df = manifest[manifest["patient_id"].isin(outer_test_ids)].copy()

    assert set(train_df["patient_id"]) == inner_train_ids
    assert set(val_df["patient_id"]) == inner_val_ids
    assert set(test_df["patient_id"]) == outer_test_ids

    train_dataset = PatientImageDataset(
        train_df,
        preprocess,
        aug_mode=TRAIN_AUG_MODE,
        deterministic=DETERMINISTIC_AUGMENTATION,
        seed=run_seed,
    )
    val_dataset = PatientImageDataset(
        val_df,
        preprocess,
        aug_mode="original",
        deterministic=True,
        seed=run_seed,
    )
    test_dataset = PatientImageDataset(
        test_df,
        preprocess,
        aug_mode="original",
        deterministic=True,
        seed=run_seed,
    )

    # 이번 검증에서 train dataset도 원본 image 수와 정확히 같아야 합니다.
    assert TRAIN_AUG_MODE == "original"
    assert len(train_dataset) == len(train_df)
    assert len(val_dataset) == len(val_df)
    assert len(test_dataset) == len(test_df)

    train_loader = DataLoader(
        train_dataset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        num_workers=NUM_WORKERS,
        pin_memory=PIN_MEMORY,
        persistent_workers=PERSISTENT_WORKERS,
    )
    val_loader = DataLoader(
        val_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=PIN_MEMORY,
        persistent_workers=PERSISTENT_WORKERS,
    )
    test_loader = DataLoader(
        test_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=PIN_MEMORY,
        persistent_workers=PERSISTENT_WORKERS,
    )

    split_audit = {
        "inner_train_ids": inner_train_ids,
        "inner_val_ids": inner_val_ids,
        "outer_test_ids": outer_test_ids,
    }
    return (
        inner_train_patients,
        inner_val_patients,
        train_loader,
        val_loader,
        test_loader,
        split_audit,
    )


def make_optimizer(model):
    head_params = []
    adapter_params = []
    other_params = []

    for name, parameter in model.named_parameters():
        if not parameter.requires_grad:
            continue
        if name.startswith("classifier."):
            head_params.append(parameter)
        elif ".adapter." in name:
            adapter_params.append(parameter)
        else:
            other_params.append(parameter)

    assert not other_params, "adapter/classifier 외 trainable parameter가 있습니다."
    groups = []
    if adapter_params:
        groups.append({"params": adapter_params, "lr": ADAPTER_LR})
    if head_params:
        groups.append({"params": head_params, "lr": HEAD_LR})

    assert adapter_params and head_params
    return torch.optim.AdamW(groups, weight_decay=WEIGHT_DECAY)


def train_internal_adapter_one_fold(fold_number, outer_train_patients, outer_test_patients):
    run_seed = SEED + fold_number * 100
    seed_everything(run_seed)

    (
        inner_train_patients,
        inner_val_patients,
        train_loader,
        val_loader,
        test_loader,
        split_audit,
    ) = make_loaders_for_fold(
        fold_number,
        outer_train_patients,
        outer_test_patients,
    )

    model, wrapped_blocks = build_internal_adapter_classifier()
    optimizer = make_optimizer(model)
    class_weight = make_class_weight(inner_train_patients)
    criterion = nn.CrossEntropyLoss(weight=class_weight)
    scaler = torch.amp.GradScaler(
        "cuda",
        enabled=(USE_AMP and torch.cuda.is_available()),
    )

    trainable_params = count_trainable_parameters(model)
    total_params = count_total_parameters(model)
    trainable_ratio = trainable_params / max(total_params, 1)
    trainable_ratio_percent = 100.0 * trainable_ratio

    print(
        f"\n[Internal Adapter fold {fold_number}] "
        f"inner train={len(inner_train_patients)}, "
        f"val={len(inner_val_patients)}, "
        f"outer test={len(outer_test_patients)}"
    )
    print(
        f"trainable params: {trainable_params:,} / {total_params:,} "
        f"({trainable_ratio_percent:.4f}%)"
    )

    best_state = None
    best_epoch = 0
    best_val_auroc = -1.0
    epochs_without_improvement = 0

    for epoch in range(1, EPOCHS + 1):
        model.train()
        loss_sum, total = 0.0, 0
        epoch_start = time.time()

        for images, labels, _ in train_loader:
            images = images.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)

            optimizer.zero_grad(set_to_none=True)
            with torch.amp.autocast(
                "cuda",
                enabled=(USE_AMP and torch.cuda.is_available()),
            ):
                logits = model(images)
                loss = criterion(logits, labels)

            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

            loss_sum += loss.item() * labels.size(0)
            total += labels.size(0)

        _, val_y_true, val_y_prob = evaluate_model(model, val_loader)
        val_metrics = calculate_metrics(val_y_true, val_y_prob, threshold=0.5)

        improved = val_metrics["auroc"] > best_val_auroc + EARLY_STOPPING_MIN_DELTA
        if improved:
            best_val_auroc = val_metrics["auroc"]
            best_epoch = epoch
            best_state = trainable_state_dict(model)
            epochs_without_improvement = 0
        else:
            epochs_without_improvement += 1

        elapsed = time.time() - epoch_start
        print(
            f"Epoch {epoch:02d}/{EPOCHS} | "
            f"loss {loss_sum / max(total, 1):.4f} | "
            f"val AUROC {val_metrics['auroc']:.4f} "
            f"AUPRC {val_metrics['auprc']:.4f} | "
            f"early {epochs_without_improvement}/{EARLY_STOPPING_PATIENCE} | "
            f"{elapsed:.1f}s"
        )

        if epochs_without_improvement >= EARLY_STOPPING_PATIENCE:
            print("Early stopping")
            break

    assert best_state is not None
    model.load_state_dict(best_state, strict=False)
    model = model.to(device)

    # Threshold calibration에는 inner validation prediction만 사용합니다.
    val_ids, val_y_true, val_y_prob = evaluate_model(model, val_loader)
    assert set(val_ids) == split_audit["inner_val_ids"]
    assert set(val_ids).isdisjoint(split_audit["outer_test_ids"])
    selected_threshold, target_reached, threshold_table = (
        choose_sensitivity_first_threshold(val_y_true, val_y_prob)
    )
    threshold_table.to_csv(
        THRESHOLD_DIR / f"fold{fold_number}_validation_thresholds.csv",
        index=False,
        encoding="utf-8-sig",
    )

    # Outer test는 threshold 확정 이후에 한 번 평가합니다.
    test_ids, test_y_true, test_y_prob = evaluate_model(model, test_loader)
    assert set(test_ids) == split_audit["outer_test_ids"]
    assert len(test_ids) == len(set(test_ids))

    test_metrics = calculate_metrics(
        test_y_true,
        test_y_prob,
        threshold=selected_threshold,
    )
    test_predictions = (np.asarray(test_y_prob) >= selected_threshold).astype(int)
    fold_oof = pd.DataFrame({
        "patient_id": test_ids,
        "target": np.asarray(test_y_true).astype(int),
        "probability": np.asarray(test_y_prob).astype(float),
        "prediction": test_predictions,
        "fold": fold_number,
        "threshold": selected_threshold,
    })
    assert set(fold_oof["patient_id"]) == split_audit["outer_test_ids"]

    checkpoint_path = CHECKPOINT_DIR / f"{TASK_NAME}_fold{fold_number}.pt"
    torch.save({
        "experiment": EXPERIMENT_NAME,
        "follow_up_status": "post-project validation of deferred ver14 pilot",
        "fold": fold_number,
        "trainable_state_dict": best_state,
        "best_epoch": best_epoch,
        "best_val_auroc": best_val_auroc,
        "selected_threshold": selected_threshold,
        "validation_target_reached": target_reached,
        "wrapped_blocks": wrapped_blocks,
        "inner_train_patient_ids": sorted(split_audit["inner_train_ids"]),
        "inner_val_patient_ids": sorted(split_audit["inner_val_ids"]),
        "outer_test_patient_ids": sorted(split_audit["outer_test_ids"]),
        "config": config_snapshot,
    }, checkpoint_path)

    result = {
        "experiment": EXPERIMENT_NAME,
        "fold": fold_number,
        "best_epoch": best_epoch,
        "val_auroc": best_val_auroc,
        "selected_threshold": selected_threshold,
        "validation_target_reached": target_reached,
        "test_patients": len(test_ids),
        "accuracy": test_metrics["accuracy"],
        "precision": test_metrics["precision"],
        "sensitivity": test_metrics["sensitivity"],
        "specificity": test_metrics["specificity"],
        "f1": test_metrics["f1"],
        "macro_f1": test_metrics["macro_f1"],
        "auroc": test_metrics["auroc"],
        "auprc": test_metrics["auprc"],
        "trainable_params": trainable_params,
        "total_params": total_params,
        "trainable_ratio": trainable_ratio,
        "trainable_ratio_percent": trainable_ratio_percent,
        "adapter_blocks": len(wrapped_blocks),
    }

    del model, optimizer, scaler, train_loader, val_loader, test_loader
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return result, fold_oof


## 9. 5-Fold 사전 검증 및 resume 실행

먼저 모든 outer test fold가 서로 겹치지 않고 전체 patient를 정확히 한 번 덮는지 확인합니다.

완료 판정은 다음 세 파일이 모두 있을 때만 인정합니다.

- fold result row
- fold별 OOF part CSV
- fold checkpoint

따라서 실행이 중단된 뒤 재실행하면 완성된 fold는 건너뛰고, 불완전한 fold부터 다시 시작합니다.


In [ ]:
def atomic_write_csv(dataframe, path):
    path = Path(path)
    temp_path = path.with_name(path.stem + ".tmp" + path.suffix)
    dataframe.to_csv(temp_path, index=False, encoding="utf-8-sig")
    temp_path.replace(path)


outer_cv = StratifiedKFold(
    n_splits=N_SPLITS,
    shuffle=True,
    random_state=SEED,
)
patient_indices = np.arange(len(patient_table))
patient_targets = patient_table["target"].to_numpy()

fold_specs = []
fold_assignment_rows = []
seen_outer_test_ids = set()

for fold_number, (outer_train_idx, outer_test_idx) in enumerate(
    outer_cv.split(patient_indices, patient_targets),
    start=1,
):
    outer_train_patients = patient_table.iloc[outer_train_idx].reset_index(drop=True)
    outer_test_patients = patient_table.iloc[outer_test_idx].reset_index(drop=True)
    outer_train_ids = set(outer_train_patients["patient_id"])
    outer_test_ids = set(outer_test_patients["patient_id"])

    assert outer_train_ids.isdisjoint(outer_test_ids)
    assert seen_outer_test_ids.isdisjoint(outer_test_ids)
    seen_outer_test_ids.update(outer_test_ids)

    for row in outer_test_patients.itertuples(index=False):
        fold_assignment_rows.append({
            "patient_id": row.patient_id,
            "target": int(row.target),
            "class_name": row.class_name,
            "outer_fold": fold_number,
        })
    fold_specs.append((fold_number, outer_train_patients, outer_test_patients))

assert seen_outer_test_ids == set(patient_table["patient_id"])
assert len(seen_outer_test_ids) == EXPECTED_PATIENTS
fold_assignments = pd.DataFrame(fold_assignment_rows).sort_values("patient_id")
assert not fold_assignments["patient_id"].duplicated().any()
atomic_write_csv(fold_assignments, FOLD_ASSIGNMENTS_PATH)

print("Outer-fold leakage audit passed.")
print(f"Unique outer test patients: {len(seen_outer_test_ids)}")
display(
    fold_assignments.groupby(["outer_fold", "target"])
    .size()
    .unstack(fill_value=0)
)

if RESULTS_PATH.exists():
    existing_results = pd.read_csv(RESULTS_PATH)
else:
    existing_results = pd.DataFrame()

result_folds = (
    set(existing_results["fold"].astype(int))
    if not existing_results.empty else set()
)
oof_part_folds = {
    fold
    for fold in FOLDS_TO_RUN
    if (OOF_PARTS_DIR / f"fold{fold}_oof.csv").exists()
}
checkpoint_folds = {
    fold
    for fold in FOLDS_TO_RUN
    if (CHECKPOINT_DIR / f"{TASK_NAME}_fold{fold}.pt").exists()
}
completed_folds = result_folds & oof_part_folds & checkpoint_folds

# 중간 저장이 불완전했던 fold row는 제거하고 해당 fold를 다시 학습합니다.
if not existing_results.empty:
    existing_results = existing_results[
        existing_results["fold"].astype(int).isin(completed_folds)
    ].copy()
all_results = existing_results.to_dict("records")

print(f"Resume 완료 folds: {sorted(completed_folds)}")

for fold_number, outer_train_patients, outer_test_patients in fold_specs:
    if fold_number not in FOLDS_TO_RUN:
        continue
    if fold_number in completed_folds:
        print(f"[SKIP] 이미 완료됨: fold {fold_number}")
        continue

    print(
        f"\n===== INTERNAL ADAPTER OUTER FOLD {fold_number} =====\n"
        f"train patients={len(outer_train_patients)}, "
        f"test patients={len(outer_test_patients)}\n"
        f"test target counts="
        f"{outer_test_patients['target'].value_counts().sort_index().to_dict()}"
    )

    result, fold_oof = train_internal_adapter_one_fold(
        fold_number,
        outer_train_patients,
        outer_test_patients,
    )
    print("Test:", result)

    # checkpoint는 train 함수에서 먼저 저장됩니다. OOF part와 result row를 이어서 저장합니다.
    atomic_write_csv(fold_oof, OOF_PARTS_DIR / f"fold{fold_number}_oof.csv")
    all_results = [
        row for row in all_results if int(row["fold"]) != fold_number
    ]
    all_results.append(result)
    atomic_write_csv(
        pd.DataFrame(all_results).sort_values("fold"),
        RESULTS_PATH,
    )
    completed_folds.add(fold_number)
    print(f"중간 결과 저장: {RESULTS_PATH}")

results_df = pd.DataFrame(all_results).sort_values("fold").reset_index(drop=True)
display(results_df)


## 10. 5-Fold summary, pooled OOF 및 Adapter Probe 비교

다음 셀은 5개 fold가 모두 완료된 경우에만 최종 파일을 생성합니다.

- `internal_adapter_5fold_results.csv`: fold별 patient-level test 결과
- `internal_adapter_5fold_summary.csv`: 5-fold mean / standard deviation
- `internal_adapter_5fold_oof_predictions.csv`: 각 patient가 held-out model에서 정확히 한 번 받은 예측
- `internal_adapter_5fold_oof_metrics.json`: pooled OOF 지표와 confusion matrix
- `internal_adapter_5fold_vs_adapter_probe.csv`: 기존 Adapter Probe와 동일 기준 비교

5-fold mean과 pooled OOF는 계산 방식이 다르므로 별도로 기록합니다.


In [ ]:
assert RESULTS_PATH.exists(), f"결과 CSV가 아직 없습니다: {RESULTS_PATH}"
results_df = pd.read_csv(RESULTS_PATH).sort_values("fold").reset_index(drop=True)
assert set(results_df["fold"].astype(int)) == set(FOLDS_TO_RUN), (
    "5개 fold가 모두 완료되지 않았습니다. 실행 셀을 재실행해 resume하세요."
)
assert results_df["fold"].nunique() == N_SPLITS

metric_columns = [
    "accuracy",
    "precision",
    "sensitivity",
    "specificity",
    "f1",
    "macro_f1",
    "auroc",
    "auprc",
    "selected_threshold",
    "best_epoch",
    "trainable_params",
    "trainable_ratio",
    "trainable_ratio_percent",
]

summary_row = {
    "experiment": EXPERIMENT_NAME,
    "folds": int(results_df["fold"].nunique()),
    "validation_target_reached_folds": int(
        results_df["validation_target_reached"].sum()
    ),
}
for metric in metric_columns:
    summary_row[f"{metric}_mean"] = float(results_df[metric].mean())
    summary_row[f"{metric}_std"] = float(results_df[metric].std(ddof=1))

summary_df = pd.DataFrame([summary_row])
atomic_write_csv(summary_df, SUMMARY_PATH)

oof_parts = []
for fold in FOLDS_TO_RUN:
    part_path = OOF_PARTS_DIR / f"fold{fold}_oof.csv"
    assert part_path.exists(), f"OOF part가 없습니다: {part_path}"
    part = pd.read_csv(part_path)
    assert set(part["fold"].astype(int)) == {fold}
    oof_parts.append(part)

oof_df = pd.concat(oof_parts, ignore_index=True).sort_values("patient_id")
assert len(oof_df) == EXPECTED_PATIENTS
assert len(oof_df) == len(patient_table)
assert not oof_df["patient_id"].duplicated().any()
assert set(oof_df["patient_id"]) == set(patient_table["patient_id"])
assert set(oof_df["fold"].astype(int)) == set(FOLDS_TO_RUN)

target_check = patient_table[["patient_id", "target"]].merge(
    oof_df[["patient_id", "target"]],
    on="patient_id",
    suffixes=("_manifest", "_oof"),
    validate="one_to_one",
)
assert (target_check["target_manifest"] == target_check["target_oof"]).all()

expected_predictions = (
    oof_df["probability"].to_numpy()
    >= oof_df["threshold"].to_numpy()
).astype(int)
assert np.array_equal(expected_predictions, oof_df["prediction"].to_numpy())
atomic_write_csv(oof_df, OOF_PREDICTIONS_PATH)

oof_metrics = calculate_metrics(
    oof_df["target"].to_numpy(),
    oof_df["probability"].to_numpy(),
    threshold=0.5,
)
# 분류 지표는 fold별 validation threshold로 저장된 prediction을 사용합니다.
oof_y_true = oof_df["target"].to_numpy().astype(int)
oof_y_pred = oof_df["prediction"].to_numpy().astype(int)
oof_cm = confusion_matrix(oof_y_true, oof_y_pred, labels=[0, 1])
tn, fp, fn, tp = [int(value) for value in oof_cm.ravel()]
pooled_thresholded = {
    "accuracy": float(accuracy_score(oof_y_true, oof_y_pred)),
    "precision": float(precision_score(oof_y_true, oof_y_pred, zero_division=0)),
    "sensitivity": float(recall_score(oof_y_true, oof_y_pred, zero_division=0)),
    "specificity": float(tn / max(tn + fp, 1)),
    "f1": float(f1_score(oof_y_true, oof_y_pred, zero_division=0)),
    "macro_f1": float(f1_score(oof_y_true, oof_y_pred, average="macro", zero_division=0)),
    "auroc": float(oof_metrics["auroc"]),
    "auprc": float(oof_metrics["auprc"]),
}
oof_payload = {
    "experiment": EXPERIMENT_NAME,
    "aggregation": "pooled patient-level OOF",
    "classification_threshold": "fold-specific threshold selected on inner validation",
    "patients": int(len(oof_df)),
    "folds": N_SPLITS,
    **pooled_thresholded,
    "confusion_matrix": [[tn, fp], [fn, tp]],
    "tn": tn,
    "fp": fp,
    "fn": fn,
    "tp": tp,
}
OOF_METRICS_PATH.write_text(
    json.dumps(oof_payload, indent=2, ensure_ascii=False),
    encoding="utf-8",
)

# Adapter Probe mean은 final comparison table, std는 실제 5-fold summary에서 읽습니다.
final_comparison_path = RESULTS_DIR / "final_model_comparison_table.csv"
adapter_summary_path = RESULTS_DIR / "non_vs_demented_biomedclip_summary.csv"
assert final_comparison_path.exists()
assert adapter_summary_path.exists()

final_comparison = pd.read_csv(final_comparison_path)
adapter_row = final_comparison[
    final_comparison["Method"] == "BiomedCLIP adapter probe"
].iloc[0]
adapter_summary = pd.read_csv(adapter_summary_path)
adapter_std_row = adapter_summary[
    adapter_summary["experiment"] == "adapter_probe"
].iloc[0]

comparison_rows = [
    {
        "model": "Adapter Probe",
        "folds": int(adapter_row["Folds"]),
        "trainable_params": float(str(adapter_row["Trainable Params"]).replace(",", "")),
        "trainable_ratio_percent": float(adapter_row["Trainable Ratio (%)"]),
        "sensitivity_mean": float(adapter_row["Sensitivity"]),
        "sensitivity_std": float(adapter_std_row["calibrated_sensitivity_std"]),
        "specificity_mean": float(adapter_row["Specificity"]),
        "specificity_std": float(adapter_std_row["calibrated_specificity_std"]),
        "precision_mean": float(adapter_row["Precision"]),
        "precision_std": float(adapter_std_row["calibrated_precision_std"]),
        "f1_mean": float(adapter_row["F1"]),
        "f1_std": float(adapter_std_row["calibrated_f1_std"]),
        "macro_f1_mean": float(adapter_row["Macro F1"]),
        "macro_f1_std": float(adapter_std_row["calibrated_macro_f1_std"]),
        "auroc_mean": float(adapter_row["AUROC"]),
        "auroc_std": float(adapter_std_row["auroc_std"]),
        "auprc_mean": float(adapter_row["AUPRC"]),
        "auprc_std": float(adapter_std_row["auprc_std"]),
    },
    {
        "model": "Internal Adapter follow-up",
        "folds": N_SPLITS,
        "trainable_params": summary_row["trainable_params_mean"],
        "trainable_ratio_percent": summary_row["trainable_ratio_percent_mean"],
        "sensitivity_mean": summary_row["sensitivity_mean"],
        "sensitivity_std": summary_row["sensitivity_std"],
        "specificity_mean": summary_row["specificity_mean"],
        "specificity_std": summary_row["specificity_std"],
        "precision_mean": summary_row["precision_mean"],
        "precision_std": summary_row["precision_std"],
        "f1_mean": summary_row["f1_mean"],
        "f1_std": summary_row["f1_std"],
        "macro_f1_mean": summary_row["macro_f1_mean"],
        "macro_f1_std": summary_row["macro_f1_std"],
        "auroc_mean": summary_row["auroc_mean"],
        "auroc_std": summary_row["auroc_std"],
        "auprc_mean": summary_row["auprc_mean"],
        "auprc_std": summary_row["auprc_std"],
    },
]
comparison_df = pd.DataFrame(comparison_rows)
atomic_write_csv(comparison_df, COMPARISON_PATH)

print("5-fold completion and OOF audit passed.")
print(f"OOF patients: {len(oof_df)} (each exactly once)")
print(f"Confusion matrix: {oof_cm.tolist()}")
display(results_df)
display(summary_df)
display(pd.DataFrame([oof_payload]))
display(comparison_df)

print("\nSaved files:")
for path in [
    RESULTS_PATH,
    SUMMARY_PATH,
    OOF_PREDICTIONS_PATH,
    OOF_METRICS_PATH,
    COMPARISON_PATH,
]:
    print(f"- {path}")


## 11. 후속 검증 해석 기준

이 실험은 Internal Adapter를 성능 최대화하기 위한 탐색이 아니라, ver14 Fold 1 결과의 재현성과 fold 안정성을 검증합니다. AUROC의 작은 차이만으로 우열을 결론내리지 않습니다.

### A. 최종 classifier 재검토 가치가 있는 경우

- Internal Adapter의 sensitivity가 높게 유지됨
- specificity와 Macro F1 손실이 작음
- sensitivity 및 주요 metric의 fold standard deviation이 Adapter Probe와 비슷하거나 더 작음

### B. Fold 1이 optimistic result였을 가능성이 큰 경우

- 일부 fold에서만 sensitivity가 매우 높음
- fold 간 standard deviation이 Adapter Probe보다 뚜렷하게 큼
- 높은 sensitivity가 specificity 급락과 함께 나타남

이 경우 기존 Adapter Probe의 안정성이 더 강한 근거가 됩니다.

### C. 기존 최종 선택을 지지하는 경우

- 5-fold mean에서 Internal Adapter가 Adapter Probe보다 전반적으로 낮음
- 또는 높은 parameter cost에도 sensitivity / Macro F1 / 안정성 이점이 없음

### 결과 반영 원칙

실제 5-fold가 완료되고 위 결과를 검토하기 전까지 다음 파일의 결론은 바꾸지 않습니다.

- `README.md`의 final selected classifier
- `results/final_model_comparison_table.csv`의 Adapter Probe final selection
- 기존 project chronology / decision log

검증 완료 후에도 본 실험은 `Follow-up validation of the deferred Internal Adapter experiment`로 기록하며, 원래 프로젝트 당시 5-fold를 수행한 것처럼 소급 서술하지 않습니다.
